In [20]:
import subprocess
subprocess.run(["pip", "install", "webdriver-manager", "--quiet"], check=True)
print("Installed.")

Installed.


In [ ]:
import pandas as pd
import os
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── 1. Load DataFrame ─────────────────────────────────────────────────────────
df = pd.read_excel("Wells_Needing_Lith_Descriptions.xlsx", sheet_name="Sheet1", dtype=str)
df["Lithology_Description"] = df["Lithology_Description"].fillna("").str.strip()
print(f"Loaded {len(df)} rows.")
print(f"  Already filled: {(df['Lithology_Description'] != '').sum()}")
print(f"  Need filling:   {(df['Lithology_Description'] == '').sum()}")

# ── 2. Chrome driver ──────────────────────────────────────────────────────────
def make_driver():
    os.environ["WDM_LOG_LEVEL"] = "0"
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--remote-debugging-port=0")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--log-level=3")
    options.add_experimental_option("excludeSwitches", ["enable-logging"])
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

# ── 3. Parse lithology table (no <th>, all <td>) ──────────────────────────────
def parse_lithology(soup):
    intervals = []
    for table in soup.find_all("table"):
        rows = table.find_all("tr")
        if len(rows) < 2:
            continue

        all_rows = []
        for row in rows:
            cells = [td.get_text(strip=True) for td in row.find_all(["td", "th"])]
            if any(cells):
                all_rows.append(cells)

        if len(all_rows) < 2:
            continue

        # Find header row containing top, bottom, description
        header_idx = None
        for i, row in enumerate(all_rows):
            joined = " ".join(row).lower()
            if "top" in joined and "bot" in joined and "desc" in joined:
                header_idx = i
                break

        if header_idx is None:
            continue

        headers = [c.lower() for c in all_rows[header_idx]]
        top_idx  = next((i for i, h in enumerate(headers) if "top"  in h), None)
        bot_idx  = next((i for i, h in enumerate(headers) if "bot"  in h), None)
        desc_idx = next((i for i, h in enumerate(headers) if "desc" in h), None)

        if None in (top_idx, bot_idx, desc_idx):
            continue

        for row in all_rows[header_idx + 1:]:
            if len(row) <= max(top_idx, bot_idx, desc_idx):
                continue
            top  = row[top_idx].strip()
            bot  = row[bot_idx].strip()
            desc = row[desc_idx].strip()

            def is_depth(x):
                return bool(x) and x.replace(".", "", 1).isdigit()

            # *** KEY FIX: description must contain at least one letter ***
            def is_real_description(x):
                return bool(x) and any(c.isalpha() for c in x)

            if is_depth(top) and is_real_description(desc):
                intervals.append({"top": top, "bottom": bot, "desc": desc})

        if intervals:
            break

    return intervals

# ── 4. Format: "0- Topsoil; 1-24 Tan Limestone; ..." ─────────────────────────
def format_lithology(intervals):
    parts = []
    for r in intervals:
        top  = r["top"].strip()
        bot  = r["bottom"].strip()
        desc = r["desc"].strip()
        if not top or not desc:
            continue
        parts.append(f"{top}-{bot} {desc}" if bot else f"{top}- {desc}")
    return "; ".join(parts)

# ── 5. Click helper — tries multiple XPath patterns ──────────────────────────
def click_element(driver, xpaths, timeout=20):
    wait = WebDriverWait(driver, timeout)
    for xpath in xpaths:
        try:
            el = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
            driver.execute_script("arguments[0].scrollIntoView(true);", el)
            el.click()
            return True
        except Exception:
            continue
    return False

# ── 6. Fetch lithology: load → View Report → Next Page → parse ───────────────
def get_lithology(url, driver, tracking=""):
    try:
        driver.get(url)
        wait = WebDriverWait(driver, 30)

        # Step 1: Click "View Report"
        view_report_xpaths = [
            "//input[@value='View Report']",
            "//button[contains(text(),'View Report')]",
            "//a[contains(text(),'View Report')]",
            "//*[contains(@value,'View Report')]",
            "//*[contains(text(),'View Report')]",
        ]
        clicked = click_element(driver, view_report_xpaths, timeout=20)
        if not clicked:
            print(f"  ⚠ Could not find 'View Report' button")
        else:
            print(f"  ✓ Clicked 'View Report'")

        # Step 2: Wait for report to render (Loading... gone)
        try:
            wait.until(EC.invisibility_of_element_located(
                (By.XPATH, "//*[contains(text(),'Loading...')]")
            ))
        except Exception:
            pass
        time.sleep(3)

        # Step 3: Click "Next Page" to get to page 2 (lithology)
        next_page_xpaths = [
            "//input[@title='Next Page']",
            "//a[@title='Next Page']",
            "//input[contains(@id,'Next')]",
            "//a[contains(@id,'Next')]",
            "//*[@title='Next Page']",
            "//*[contains(@id,'_Next')]",
            "//*[contains(@name,'Next')]",
            "//*[contains(@id,'NavigationNext')]",
        ]
        clicked2 = click_element(driver, next_page_xpaths, timeout=15)
        if not clicked2:
            print(f"  ⚠ Could not find 'Next Page' button — trying page 2 input")
            # Try typing "2" into the page number box
            try:
                page_input = driver.find_element(By.XPATH,
                    "//input[@type='text' and (contains(@id,'CurrentPage') or contains(@id,'Page') or contains(@title,'page number'))]"
                )
                page_input.clear()
                page_input.send_keys("2")
                from selenium.webdriver.common.keys import Keys
                page_input.send_keys(Keys.RETURN)
                print(f"  ✓ Typed page 2 in page number input")
            except Exception:
                print(f"  ⚠ Could not navigate to page 2")
        else:
            print(f"  ✓ Clicked 'Next Page'")

        # Step 4: Wait for page 2 content to load
        time.sleep(4)

        # Step 5: Parse
        soup = BeautifulSoup(driver.page_source, "lxml")
        intervals = parse_lithology(soup)

        if not intervals:
            with open(f"debug_failed_{tracking}.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)

        return intervals

    except Exception as e:
        print(f"  Selenium error: {e}")
        return []

# ── 7. Main loop ──────────────────────────────────────────────────────────────
# ── 7. Main loop ──────────────────────────────────────────────────────────────
filled = failed = skipped = 0
AUTOSAVE_EVERY = 10  # save every 10 rows processed
rows_since_save = 0
output_file = "Wells_Needing_Lith_Descriptions_filled2.xlsx"
driver = make_driver()
print("\nStarting...\n")

try:
    for idx, row in df.iterrows():
        lith = row.get("Lithology_Description", "")
        if lith and lith.lower() not in ("nan", "none", ""):
            skipped += 1
            continue

        url      = str(row.get("Report_URL", "")).strip()
        tracking = str(row.get("WellReportTrackingNumber", "")).strip()

        if not url or url.lower() in ("nan", "none", ""):
            print(f"Row {idx} (#{tracking}) — no URL.")
            failed += 1
            continue

        print(f"Processing row {idx}, Tracking #{tracking}")
        intervals = get_lithology(url, driver, tracking)

        if not intervals:
            print(f"  ✗ No lithology found.")
            failed += 1
        else:
            lith_str = format_lithology(intervals)
            df.at[idx, "Lithology_Description"] = lith_str
            filled += 1
            print(f"  ✓ {lith_str[:120]}{'...' if len(lith_str) > 120 else ''}")

        # ── Autosave every N rows ──────────────────────────────────────────
        rows_since_save += 1
        if rows_since_save >= AUTOSAVE_EVERY:
            df.to_excel(output_file, sheet_name="Sheet1", index=False)
            print(f"  💾 Autosaved ({filled} filled so far)")
            rows_since_save = 0

except KeyboardInterrupt:
    print("\n⚠ Interrupted by user — saving progress...")

finally:
    driver.quit()
    print("Browser closed.")

# Final save
df.to_excel(output_file, sheet_name="Sheet1", index=False)
print(f"\nSaved: {output_file}")
print(f"  ✓ Filled: {filled} | ↷ Had data: {skipped} | ✗ Failed: {failed}")

# ── 8. Save ───────────────────────────────────────────────────────────────────
output_file = "Wells_Needing_Lith_Descriptions_filled.xlsx"
df.to_excel(output_file, sheet_name="Sheet1", index=False)
print(f"Saved: {output_file}")
print(f"  ✓ Filled: {filled} | ↷ Had data: {skipped} | ✗ Failed: {failed}")

Loaded 166 rows.
  Already filled: 0
  Need filling:   166

Starting...

Processing row 0, Tracking #309816
  ✓ Clicked 'View Report'
  ✓ Clicked 'Next Page'
  ✓ 0-1 Red Clay; 1-15 White Rock; 15-25 Red Clay; 25-180 Black Rock; 180-220 Brown Rock; 220-300 White Rock
Processing row 1, Tracking #310067
  ✓ Clicked 'View Report'
  ✓ Clicked 'Next Page'
  ✓ 1-22 brown limestone; 22-46 gray shale & clay; 46-61 red & gray clay; 61-66 red shale & clay; 66-84 red limestone with r...
Processing row 2, Tracking #310107
  ✓ Clicked 'View Report'
  ✓ Clicked 'Next Page'
  ✓ 0-10 Red Sandy Lime; 10-35 Sand & Gravel; 35-80 Gray Clay; 80-160 Brown w/ Black Streaks of Granite; 160-165 Black Grani...
Processing row 3, Tracking #311600
  ✓ Clicked 'View Report'
  ✓ Clicked 'Next Page'
  ✓ 0-160 exsisting borehole; 160-170 multi-colored limestones (gravel); 170-175 yellow limestone; 175-190 yellow clay; 190-...
Processing row 4, Tracking #314180
  ✓ Clicked 'View Report'
  ✓ Clicked 'Next Page'
  ✓ 0-5 t